# `temperature_scaling` -- Single-Parameter Probability Calibration

**Ported from `project_hrishav/calibration.py`.** This is one of the two calibration approaches kept in the merged project, deliberately *not* unified with `src/models.py`'s `CalibratedClassifierCV` (isotonic) approach -- see the module docstring below for why forcing a shared abstraction here would add complexity for no benefit.

**The idea (Guo et al. 2017, "On Calibration of Modern Neural Networks"):** a model's raw probabilities can be systematically too confident or too cautious. Temperature scaling fixes this with a *single* learned scalar `T`:

```
logit(p) = log(p / (1 - p))
p_cal    = sigmoid(logit(p) / T)
```

- `T > 1`: softens overconfident predictions (pulls them toward 0.5)
- `T < 1`: sharpens overcautious predictions (pushes them toward 0/1)
- `T = 1`: no change -- the model was already well-calibrated

Because it's a single number fit on a held-out validation set, it **cannot overfit** in any meaningful sense, and because it's a monotonic transform of the logit, it **never changes AUC/ranking** -- only how the probabilities are spread out.

In [1]:
import numpy as np
from scipy.optimize import minimize_scalar

_EPS = 1e-7

## `_logit()` / `_sigmoid()` -- private math helpers

`_logit` is the inverse of the sigmoid function: it maps a probability in (0,1) back to the real-number line, where scaling by `1/T` actually behaves the way we want. Values are clipped to `[_EPS, 1-_EPS]` first, since `log(0)` and `log(1/0)` are undefined -- a probability of exactly 0.0 or 1.0 would otherwise crash this function.

`_sigmoid` is the standard logistic function, converting a real-valued logit back into a (0,1) probability.

In [2]:
def _logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(p, _EPS, 1.0 - _EPS)
    return np.log(p / (1.0 - p))

In [3]:
def _sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

## `TemperatureScaler` class

Familiar scikit-learn-style `fit`/`transform`/`fit_transform` interface, but internally there's only one parameter to learn: `self.T`.

**`fit()`** converts the validation probabilities to logits once up front, then uses `scipy.optimize.minimize_scalar` (bounded search between T=0.05 and T=20) to find the value of `T` that **minimises negative log-likelihood** on the validation labels -- i.e. the T that makes the rescaled probabilities the best possible fit to what actually happened. `T<=0` is guarded against by returning an enormous loss (`1e18`), which the bounded optimizer will never select anyway, but makes the objective function safe to call with any float.

**`transform()`** applies the already-fitted `T` to any new batch of probabilities (typically the test set).

**`fit_transform()`** is a convenience that does both in one call, returning the *calibrated validation-set* probabilities (useful for inspecting how well the fit worked before applying it to test data).

In [4]:
class TemperatureScaler:
    """
    Fit a single scalar temperature T on validation probabilities.

    Usage
    -----
        scaler = TemperatureScaler()
        scaler.fit(val_probs, val_labels)
        cal_probs = scaler.transform(test_probs)
    """

    def __init__(self) -> None:
        self.T: float = 1.0

    def fit(self, probs_val: np.ndarray, y_val: np.ndarray) -> "TemperatureScaler":
        probs_val = np.asarray(probs_val, dtype=np.float64)
        y_val = np.asarray(y_val, dtype=np.float64)
        logits = _logit(probs_val)

        def _nll(T: float) -> float:
            if T <= 0:
                return 1e18
            p = _sigmoid(logits / T)
            p = np.clip(p, _EPS, 1.0 - _EPS)
            return -np.mean(y_val * np.log(p) + (1.0 - y_val) * np.log(1.0 - p))

        result = minimize_scalar(_nll, bounds=(0.05, 20.0), method="bounded",
                                  options={"xatol": 1e-4})
        self.T = float(result.x)
        return self

    def transform(self, probs: np.ndarray) -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return _sigmoid(_logit(probs) / self.T)

    def fit_transform(self, probs_val: np.ndarray, y_val: np.ndarray) -> np.ndarray:
        self.fit(probs_val, y_val)
        return self.transform(probs_val)

## `compare_calibration()` -- before/after reporting helper

A convenience function that runs the full calibration workflow and reports every relevant metric before and after, in one dict: fit `T` on the validation set, apply it to the test set, then compute Brier score, log-loss, AUC, and ECE for *both* the raw and calibrated test-set probabilities side by side.

**AUC is included specifically to demonstrate it doesn't change** -- `auc_raw` and `auc_cal` should always come out equal (up to floating-point noise), which is the whole point of using a monotonic transform for calibration rather than something that could reorder predictions.

**One deliberate change from the original `project_hrishav` code**: the ECE computation here calls `src.metrics.ece` (already ported and tested as part of this merge) instead of hrishav's own `evaluate.expected_calibration_error`, which wasn't ported into this project -- this keeps the module self-contained with no dependency on unported hrishav code.

In [5]:
def compare_calibration(
    name: str,
    probs_val: np.ndarray, y_val: np.ndarray,
    probs_test: np.ndarray, y_test: np.ndarray,
    n_bins: int = 10,
) -> dict:
    """
    Fit temperature on val, apply to test, return before/after metrics.

    Returns a dict with: T, brier_raw, brier_cal, logloss_raw, logloss_cal,
    auc_raw, auc_cal, ece_raw, ece_cal.
    """
    from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss

    from src.metrics import ece

    scaler = TemperatureScaler().fit(probs_val, y_val)
    cal_test = scaler.transform(probs_test)

    return {
        "model": name,
        "T": round(scaler.T, 4),
        "brier_raw": round(brier_score_loss(y_test, probs_test), 4),
        "brier_cal": round(brier_score_loss(y_test, cal_test), 4),
        "logloss_raw": round(log_loss(y_test, np.clip(probs_test, _EPS, 1 - _EPS)), 4),
        "logloss_cal": round(log_loss(y_test, np.clip(cal_test, _EPS, 1 - _EPS)), 4),
        "auc_raw": round(roc_auc_score(y_test, probs_test), 4),
        "auc_cal": round(roc_auc_score(y_test, cal_test), 4),
        "ece_raw": round(ece(np.asarray(y_test), np.asarray(probs_test), n_bins=n_bins), 4),
        "ece_cal": round(ece(np.asarray(y_test), np.asarray(cal_test), n_bins=n_bins), 4),
    }